In [15]:
import sys
import os
current_path = os.path.abspath('')
data_path = '/'.join(current_path.split('/')[:-1]) + '/data/'
script_path = '/'.join(current_path.split('/')[:-1]) + '/scripts/'
sys.path.append(script_path)
sys.path.append('/home/fkunneman/.local/lib/python3.10/site-packages/')
sys.path.append('/usr/lib/python3/dist-packages/')

In [16]:
import torch
import gc
import importlib
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from sentence_transformers import SentenceTransformer
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_huggingface.llms import HuggingFacePipeline
from langchain_community.vectorstores import Chroma
import chromadb
from chromadb.api.types import EmbeddingFunction, Documents, Embeddings

import instruction_agent

In [3]:
### Load models for chatbot and rag in memory
with open('/home/fkunneman/hf.txt') as fin:
    hf = fin.read().strip()
os.environ["HF_TOKEN"] = hf
torch.cuda.empty_cache()
gc.collect()
# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained('BramVanroy/GEITje-7B-ultra', use_fast=True)
tokenizer.pad_token = tokenizer.eos_token # Ensure padding token is set
# Initialize language model
chatmodel = AutoModelForCausalLM.from_pretrained('BramVanroy/GEITje-7B-ultra', dtype=torch.bfloat16, trust_remote_code=True, device_map="cuda")
#chatmodel = AutoModelForCausalLM.from_pretrained('robinsmits/Qwen1.5-7B-Dutch-Chat', dtype=torch.bfloat16, trust_remote_code=True, device_map="cuda")     
# set pipeline
pipe = pipeline(
    "text-generation",
    model=chatmodel,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=250, # Limit the number of generated tokens for concise responses
    do_sample=True, # Enable sampling for more varied responses
    temperature=0.3 # Control creativity
)
llm = HuggingFacePipeline(pipeline=pipe)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [4]:
chroma_client = chromadb.Client()
#embedding_model = SentenceTransformer('jegormeister/bert-base-dutch-cased')
collection = chroma_client.create_collection(
    name="instruction_db5",
    configuration={
        "hnsw": {
            "space": "cosine",
            "ef_construction": 30
        }
    }
)

In [17]:
instructions_travel = data_path + 'opslag_inclusieve_spraakassistent_project/instructions_ov_stripped.csv'
instructions_passport = data_path + 'opslag_inclusieve_spraakassistent_project/instructions_paspoort_stripped_v2.csv'
pat = data_path + 'opslag_inclusieve_spraakassistent_project/patterns_v2.csv'
qa_travel = data_path + 'opslag_inclusieve_spraakassistent_project/Vraag_antwoord_ov_v5.csv'
qa_passport = data_path + 'opslag_inclusieve_spraakassistent_project/Vraag_antwoord_paspoort.csv'
nav = data_path + 'opslag_inclusieve_spraakassistent_project/navigation.csv'


In [18]:
importlib.reload(instruction_agent)
assistant = instruction_agent.InstructAgent(llm,'passport')
assistant.prepare_patterns(pat)
assistant.prepare_instructions(instructions_travel,'travel')
assistant.prepare_instructions(instructions_passport,'passport')
assistant.setup_rag(collection,[['qa','travel',qa_travel],['qa','passport',qa_passport],['nav',nav]])

In [ ]:
assistant.chat()

Hallo! Ik ga je helpen met het maken van een afspraak voor een paspoort. Je kan altijd een vraag stellen. Om te beginnen, zeg 'ik wil starten'.


You:  start


{'ids': [['a53cd6b8-9b9f-44b9-90fc-db94840259cc', '13039669-02ed-45f9-af9b-48e18e8cd113', '9c116c3f-7072-41cc-94ec-d4e0f6898c3b', 'b555689a-a8fc-4407-96b3-1abc1902a2ae', '001ee374-99c6-4ed0-a7df-0131a154c451', '38339a0f-4a8a-476c-aac8-b867bfc193d4', 'a21f26d1-2a99-46c5-becc-0e1adf6ae4fc', '63d0d9b6-2490-400c-9bb3-eb8a45ee1cc8', '19696a4b-a791-40e9-ab35-a635d88a8d6c', 'bd5c329b-41e3-4b39-8597-b80a94f851d4']], 'embeddings': None, 'documents': [['Start', 'Start', 'Start', 'Start', 'Start', 'Quit', 'Quit', 'Quit', 'Quit', 'Quit']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'type': 'nav', 'action': 'start', 'step_context': 'all'}, {'step_context': 'all', 'type': 'nav', 'action': 'start'}, {'type': 'nav', 'action': 'start', 'step_context': 'all'}, {'step_context': 'all', 'action': 'start', 'type': 'nav'}, {'step_context': 'all', 'action': 'start', 'type': 'nav'}, {'step_context': 'all', 'type': 'nav', 'action': 'Done'}, {'action': 'Done'

You:  stop


{'ids': [['26e19eed-de7f-45f0-89a8-44d10306014d', '9611d595-ee2d-4717-ae38-3b9fd4b65efe', 'dec6cb66-8ef1-43aa-bde1-ca5090e0c0c7', '81e03e90-ed4a-466e-a6e3-2f41ff9154b7', '036e8f66-f830-4e3b-8f2f-b3e05ce90406', '38339a0f-4a8a-476c-aac8-b867bfc193d4', 'a21f26d1-2a99-46c5-becc-0e1adf6ae4fc', '63d0d9b6-2490-400c-9bb3-eb8a45ee1cc8', '19696a4b-a791-40e9-ab35-a635d88a8d6c', 'bd5c329b-41e3-4b39-8597-b80a94f851d4']], 'embeddings': None, 'documents': [['Stop', 'Stop', 'Stop', 'Stop', 'Stop', 'Quit', 'Quit', 'Quit', 'Quit', 'Quit']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'action': 'Done', 'step_context': 'all', 'type': 'nav'}, {'step_context': 'all', 'type': 'nav', 'action': 'Done'}, {'type': 'nav', 'step_context': 'all', 'action': 'Done'}, {'step_context': 'all', 'action': 'Done', 'type': 'nav'}, {'action': 'Done', 'step_context': 'all', 'type': 'nav'}, {'action': 'Done', 'type': 'nav', 'step_context': 'all'}, {'action': 'Done', 'step_co